In [74]:
import numpy as np,pandas as pd ,seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings('ignore')

first of all load our data

In [75]:
df = pd.read_csv('Data.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


check NAN

In [76]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [77]:
df = df.drop(columns=['gender','customerID'])
df.head()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


now we will seperate the numeric and categorical columns

In [78]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(np.mean(df['TotalCharges']))
numeric_columns = ['MonthlyCharges','tenure','TotalCharges']

now we speperated the numeric but before going to seperate the categorical cols we first clean the up 

In [79]:
un_cleaned_catagorical = []
for col in df.columns:
    if df[col].dtype == object or df[col].dtype == 'string':
        if df[col].nunique() > 2:
            un_cleaned_catagorical.append(col)
to_clean = ['MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
for col in to_clean:
    df[col] = df[col].map({'No internet service': 'No','No':'No','Yes':'Yes','No phone service':'No'})


In [80]:
df['Churn'] = df['Churn'].map({'Yes':1,'No':0})
x = df.drop('Churn',axis=1)
y= df['Churn']

In [81]:
x['SeniorCitizen'] = x['SeniorCitizen'].map({0:'No',1:'Yes'})
to_Label_encode = []
to_onhot_encode = []
to_scale = numeric_columns
for col in x.columns:
    if x[col].nunique() == 2:
        to_Label_encode.append(col)
    elif x[col].nunique() == 3 or x[col].nunique() == 4:
        to_onhot_encode.append(col)
print(to_Label_encode,to_onhot_encode)

['SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling'] ['InternetService', 'Contract', 'PaymentMethod']


no we do our furthure splits and make pipeline

In [82]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=42)

In [83]:
Preprocessor =  ColumnTransformer([
    ('scaler', StandardScaler(), to_scale),
    ('hot_encoder', OneHotEncoder(handle_unknown='ignore'), to_onhot_encode),
    ('binary_encoder', OneHotEncoder(handle_unknown='ignore', drop='if_binary'), to_Label_encode)
])
models = {
    'LogisticRegression':LogisticRegression(solver='liblinear',max_iter=6000),
    'KNN':KNeighborsClassifier(),
    'SVM': SVC(class_weight='balanced'),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier()
}
param_grids = {
    'LogisticRegression': {
        'C': [ 10,20,30],
        'penalty': ['l1', 'l2']
    },
    'KNN': {
        'n_neighbors':[21,23,29],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    },
    'NaiveBayes': {
        'var_smoothing': [1e-9, 1e-8, 1e-7] 
    },
    'SVM': {
        'C': [0.1, 1, 10, 100],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'DecisionTree': {
        'criterion': ['gini', 'entropy'],
        'max_depth': [None, 5, 10, 15, 20],
        'min_samples_split':[2,5,10],
        'min_samples_leaf': [1, 2, 4]
    }
}

grid search cv 

In [ ]:
best_models = {}
for model_name in models:
    print(f'filhal m{model_name }trian ho rha he ...........')
    my_pipeline = Pipeline([
        ('preprocessor',Preprocessor),
        ('model',models[model_name])
    ])
    current_grid = {}
    for prams, val in param_grids[model_name].items():
        current_grid[f'model__{prams}'] = val
    search = GridSearchCV(
        estimator=my_pipeline,
        param_grid=current_grid,
        cv=5,
        n_jobs=-1,
    )
    search.fit(x_train, y_train)
    best_models[model_name] = search.best_estimator_
    print(f"Optimal Parameters for {model_name}: {search.best_params_}")
    print(f"Highest CV Score: {search.best_score_:.4f}\n")


filhal mLogisticRegressiontrian ho rha he ...........


AttributeError: 'LogisticRegression' object has no attribute 'items'

now we see the score

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score

print("======= FINAL TEST SCORES =======")

for model_name, trained_model in best_models.items():

    prediction = trained_model.predict(x_test)

    print(f"\n{model_name}")
    print("-" * 30)
    print("Accuracy :", accuracy_score(y_test, prediction))
    print("Precision:", precision_score(y_test, prediction))
    print("Recall   :", recall_score(y_test, prediction))
    print("F1 Score :", f1_score(y_test, prediction))